# Linear & Logistic Regression from Scratch with Gradient Descent

### Manual Train-Test Splits, Loss Functions, Feature Scaling, and Optimization

This notebook implements the core mechanics of **linear regression and logistic regression without using scikit-learn estimators**.

Two educational examples are included:

1. **Wine quality** — linear-regression building blocks and gradient-descent implementation
2. **Titanic survival** — logistic regression with preprocessing, standardization, cross-entropy loss, and gradient descent

**Data availability:** The original Wine Quality and Titanic CSV files are not included in this portfolio archive. Stored tables and model outputs are preserved from the original execution.

# Part I — Wine Quality Linear Regression

## 1. Data Preparation

Red and white wine-quality files are loaded, tagged by wine color, and concatenated. The stored dataset contains physicochemical predictors together with a numeric `quality` target.

In [2]:
# Import Necessary Libraries
import pandas as pd
import numpy as np

In [3]:
### INSERT CODE FOR IMPORT
red_wine = pd.read_csv("winequality-red.csv", sep=';')
white_wine = pd.read_csv("winequality-white.csv", sep=';')

red_wine["wine_color"] = "red"
white_wine["wine_color"] = "white"

wine_df = pd.concat([red_wine, white_wine], ignore_index=True)
wine_df.head()


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,wine_color
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,red
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5,red
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5,red
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6,red
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,red


## 2. Manual 80/20 Split

Rather than using a library splitting utility, the combined data is shuffled with a fixed random state and partitioned by row index.

The archived run contains:

- **5,197 training observations**
- **1,300 test observations**

In [4]:
wine_df["wine_color"] = wine_df["wine_color"].map({"red": 0, "white": 1})
wine_data_shuffled = wine_df.sample(frac=1, random_state=42).reset_index(drop=True)

# Split into training and test sets (80/20)
split_index = int(0.8 * len(wine_data_shuffled))
train_data = wine_data_shuffled[:split_index]
test_data = wine_data_shuffled[split_index:]

print("Training set size:", len(train_data))
print("Test set size:", len(test_data))
train_data.head()

Training set size: 5197
Test set size: 1300


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,wine_color
0,7.0,0.17,0.74,12.8,0.045,24.0,126.0,0.99420,3.26,0.38,12.2,8,1
1,7.7,0.64,0.21,2.2,0.077,32.0,133.0,0.99560,3.27,0.45,9.9,5,0
2,6.8,0.39,0.34,7.4,0.020,38.0,133.0,0.99212,3.18,0.44,12.0,7,1
3,6.3,0.28,0.47,11.2,0.040,61.0,183.0,0.99592,3.12,0.51,9.5,6,1
4,7.4,0.35,0.20,13.9,0.054,63.0,229.0,0.99888,3.11,0.50,8.9,6,1


## 3. Sum of Squared Errors

The linear-regression objective is built around the Sum of Squared Errors:

\[
SSE=\sum_{i=1}^{n}(\hat y_i-y_i)^2.
\]

A small assertion verifies the helper against a known example.

In [5]:
def calculate_SSE(predicted_values, actual_values):
    predicted_values = np.array(predicted_values)
    actual_values = np.array(actual_values)
    errors = predicted_values - actual_values
    squared_errors = errors ** 2
    sse = np.sum(squared_errors)
    return sse


In [6]:
## ASSERT DO NOT DELETE
predicted = [2, 3, 4]
actual = [1, 5, 2]
expected_sse = 9
assert calculate_SSE(predicted, actual) == expected_sse

## 4. Weight Initialization

The model begins with zero-valued coefficients plus a bias term.

In [7]:
def initialize_weights(X_columns):
    num_weights = len(X_columns) + 1  # +1 for the bias term
    return np.zeros(num_weights)


## 5. Gradient-Descent Linear Regression

The custom training routine:

1. extracts predictors and the `quality` target;
2. inserts a bias column;
3. initializes all weights to zero;
4. computes predictions and SSE;
5. computes the gradient from residual errors; and
6. updates the weights until the SSE change is sufficiently small or the iteration limit is reached.

The original notebook defines this implementation but **does not execute and evaluate the wine-quality linear-regression model afterward**. No performance metric is therefore added or inferred in this portfolio version.

In [8]:
def linear_regression(dataset, target_column='quality', learning_rate=0.01, max_iterations=1000):
    X_columns = [col for col in dataset.columns if col != target_column]
    weights = initialize_weights(X_columns)

    X = dataset[X_columns].values
    Y = dataset[target_column].values

    # Add a column of 1s to X for the bias term
    X = np.c_[np.ones(X.shape[0]), X]

    prev_sse = float('inf')
    for iteration in range(max_iterations):
        predictions = X.dot(weights)
        errors = predictions - Y

        sse = calculate_SSE(predictions, Y)

        # Stopping condition: less than 1% SSE change
        if abs(prev_sse - sse) < 0.01 * prev_sse:
            print(f"Stopping early at iteration {iteration} | SSE: {sse:.4f}")
            break
        prev_sse = sse

        # Gradient descent weight update
        gradient = X.T.dot(errors) / len(X)
        weights -= learning_rate * gradient

    return weights

# Part II — Titanic Logistic Regression

## 6. Data-Provenance Note

The original exercise loads Kaggle's `train.csv`, `test.csv`, and `gender_submission.csv`. It merges `gender_submission.csv` onto the Kaggle test rows to create a `Survived` value and then concatenates those rows with the official training data.

That distinction matters: **`gender_submission.csv` is a sample submission, not the hidden ground-truth labels for Kaggle's test set.** Consequently, the later 79.77% accuracy is an internal educational metric on a mixed dataset that includes sample-submission labels. It should **not** be interpreted as a genuine Kaggle test-set score.

The original code and stored output are preserved below for transparency.

In [9]:
gender_submission = pd.read_csv("gender_submission.csv")
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

# Merge test with gender_submission on PassengerId
merged_df = pd.merge(test, gender_submission, on="PassengerId")
merged_df.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Survived
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q,0
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S,1
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q,0
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S,0
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S,1


In [10]:
titanic = pd.concat([train, merged_df], ignore_index=True)

print(f"Combined dataset shape: {titanic.shape}")
titanic.head()

Combined dataset shape: (1309, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## 7. Preprocessing

The modeling table retains:

- passenger class;
- sex;
- age;
- fare;
- siblings/spouses aboard; and
- parents/children aboard.

Missing Age and Fare values are filled with medians, while passenger class and sex are one-hot encoded.

In [11]:
titanic_clean = titanic[['Survived', 'Pclass', 'Sex', 'Age', 'Fare', 'SibSp', 'Parch']].copy()

# Handle missing values:
# Fill missing Age and Fare values with the median
titanic_clean['Age'] = titanic_clean['Age'].fillna(titanic_clean['Age'].median())
titanic_clean['Fare'] = titanic_clean['Fare'].fillna(titanic_clean['Fare'].median())

# One-Hot Encode 'Pclass' and 'Sex'
# Use drop_first=True to avoid the dummy variable trap.
titanic_clean = pd.get_dummies(titanic_clean, columns=['Pclass', 'Sex'], drop_first=True, dtype=int)
print("Cleaned & One-Hot Encoded Titanic Dataset:")
display(titanic_clean.head())

Cleaned & One-Hot Encoded Titanic Dataset:


,Survived,Age,Fare,SibSp,Parch,Pclass_2,Pclass_3,Sex_male
0,0,22.0,7.2500,1,0,0,1,1
1,1,38.0,71.2833,1,0,0,0,0
2,1,26.0,7.9250,0,0,0,1,0
3,1,35.0,53.1000,1,0,0,0,0
4,0,35.0,8.0500,0,0,0,1,1


## 8. Manual 80/20 Split

The cleaned combined dataset is shuffled and manually divided into:

- **1,047 training observations**
- **262 held-out observations**

In [12]:
# Shuffle the cleaned Titanic data
cleaned_data = titanic_clean.sample(frac=1, random_state=1125).reset_index(drop=True)

# Split index for 80%
split_index = int(0.8 * len(cleaned_data))

# Create training and test sets
train_data = cleaned_data[:split_index]
test_data  = cleaned_data[split_index:]

print("Training set size:", len(train_data))
print("Test set size:", len(test_data))
display(train_data.head())


Training set size: 1047
Test set size: 262


,Survived,Age,Fare,SibSp,Parch,Pclass_2,Pclass_3,Sex_male
0,1,18.0,13.0000,1,1,1,0,0
1,0,20.0,13.8625,0,0,1,0,1
2,1,22.0,151.5500,0,0,0,0,0
3,0,31.0,28.5375,0,0,0,0,1
4,0,28.0,7.2250,0,0,0,1,1


## 9. Logistic-Regression Components

The implementation defines:

- the sigmoid function;
- binary cross-entropy / log loss;
- SSE as a supporting utility; and
- zero-valued weight initialization.

For a linear score \(z\), the sigmoid converts the value to a probability:

\[
\sigma(z)=\frac{1}{1+e^{-z}}.
\]

In [13]:
def sigmoid(z):
    """Compute the sigmoid function on a numeric input."""
    z = np.array(z, dtype=np.float64)  # ensure numeric type
    return 1 / (1 + np.exp(-z))

def calculate_log_loss(y_true, y_pred, epsilon=1e-8):
    """Compute the binary cross-entropy loss (log loss) for predictions."""
    y_true = np.array(y_true)
    y_pred = np.clip(np.array(y_pred), epsilon, 1 - epsilon)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

def calculate_SSE(predicted_values, actual_values):
    """Compute the Sum of Squared Errors (SSE)."""
    return np.sum((np.array(actual_values) - np.array(predicted_values)) ** 2)

def initialize_weights(X_columns):
    """
    Initialize weights to zeros.
    Note: here the number of weights equals the number of columns in X.
    (Because we inserted the bias column already.)
    """
    return np.zeros(len(X_columns))

## 10. Gradient-Descent Training

Training standardizes predictors using statistics calculated from the training split, adds a bias term, and optimizes binary cross-entropy using manually accumulated gradients.

The stored run stopped at **iteration 215** with a training loss of approximately **0.4921**.

In [14]:
def logistic_regression(dataset, learning_rate=0.01, max_iterations=1000):
    """
    Train a logistic regression model using gradient descent with feature scaling.
    
    The function first normalizes the features (except for the target) using the training
    set's mean and standard deviation, inserts a bias column, then applies gradient descent.
    
    Parameters:
      - dataset: DataFrame with the target 'Survived' and features.
      - learning_rate: Step size for gradient descent.
      - max_iterations: Maximum training iterations.
      
    Returns:
      - weights: Learned weights (including bias as the first weight).
    """
    # Separate target and features.
    y = dataset['Survived']
    X = dataset.drop(columns=['Survived'])
    
    # Compute means and standard deviations on training features for normalization.
    means = X.mean()
    stds  = X.std()
    X = (X - means) / stds  # apply normalization
    
    # Store these statistics for test set processing.
    logistic_regression.means = means
    logistic_regression.stds  = stds
    
    # Insert bias column.
    X.insert(0, 'bias', 1)
    
    # Initialize weights (length equals number of columns in X).
    weights = initialize_weights(X.columns)
    
    prev_loss = float('inf')
    loss_list = []
    
    # Training loop.
    for iteration in range(max_iterations):
        loss = 0
        gradients = np.zeros_like(weights)
        
        # Loop through each training sample (non-vectorized version for clarity).
        for i in range(len(X)):
            x_i = list(X.iloc[i])    # convert row to list
            y_i = y.iloc[i]
            z = np.dot(weights, x_i)
            prediction = sigmoid(z)
            
            # Compute cross-entropy loss for the sample.
            loss += - (y_i * np.log(prediction + 1e-8) + (1 - y_i) * np.log(1 - prediction + 1e-8))
            
            # Accumulate gradient: (prediction - actual) * x_i.
            gradients += (prediction - y_i) * np.array(x_i)
        
        loss /= len(X)
        loss_list.append(loss)
        
        # Convergence: If the relative change in loss is less than 0.1%, stop training.
        if abs(prev_loss - loss) < 0.001 * prev_loss:
            print(f"Stopped at iteration {iteration}, Loss: {loss:.4f}")
            break
        prev_loss = loss
        
        # Update weights using the averaged gradient.
        weights -= learning_rate * (gradients / len(X))
    
    logistic_regression.loss_list = loss_list
    return weights

# Train logistic regression on Titanic training data.
weights = logistic_regression(train_data, learning_rate=0.01, max_iterations=1000)

Stopped at iteration 215, Loss: 0.4921


## 11. Held-Out Evaluation

Using the archived combined-data split, the stored model reports:

- **Log loss:** 0.5329
- **Accuracy:** 79.77%

These values demonstrate that the custom logistic-regression implementation produces coherent probability and class predictions, but—because of the label-provenance issue described above—they should be treated as **algorithm-demonstration metrics rather than external benchmark performance**.

In [15]:
X_test = test_data.drop(columns=['Survived'])
Y_test = test_data['Survived']

X_test_std = (X_test - logistic_regression.means) / logistic_regression.stds
# Insert bias column.
X_test_std.insert(0, 'bias', 1)

X_test_array = X_test_std.to_numpy()

predictions_prob = sigmoid(np.dot(X_test_array, weights))

test_log_loss = calculate_log_loss(Y_test, predictions_prob)

predictions_class = (predictions_prob >= 0.5).astype(int)
accuracy = np.mean(predictions_class == Y_test.values)

print("\nFinal Log Loss on Test Set: {:.4f}".format(test_log_loss))
print("Test Accuracy: {:.2f}%".format(accuracy * 100))



Final Log Loss on Test Set: 0.5329
Test Accuracy: 79.77%


## 12. Key Takeaways and Limitations

### What the notebook demonstrates

- manual train-test splitting;
- construction of regression loss functions;
- zero initialization and bias handling;
- gradient descent without a library estimator;
- training-set standardization;
- probabilistic binary classification; and
- evaluation through log loss and accuracy.

### What would be improved in a production version

- evaluate the wine linear-regression implementation on a genuine held-out set;
- scale wine predictors before gradient descent;
- keep Kaggle test data unlabeled and evaluate Titanic only against verified labels;
- use cross-validation or a dedicated validation split for tuning;
- vectorize the per-row logistic-regression training loop for efficiency; and
- compare the from-scratch implementations against a trusted library baseline.

This notebook is therefore best presented as a **from-scratch optimization exercise**, not as a benchmark claim.